In [ ]:
import xarray as xr
import numpy as np
from scipy.fft import fft, ifft, fftfreq, fft2, ifft2
import pandas as pd

In [ ]:
import os
import gc

# KELVIN WAVE

In [ ]:
def space_time_filter(time_low_cut, time_high_cut,
                      space_low_cut, space_high_cut,
                      data, tim_int, space_int, ):

  
    nt, _, _, _, nl = data.shape
  
    ftime = fftfreq(nt, tim_int)            
    flon = fftfreq(nl, space_int)        
                     
    fft_data = fft2(data, axes=(0, 4))

    time_mask = ((np.abs(ftime) >= time_low_cut) &
                 (np.abs(ftime) <= time_high_cut) & (ftime > 0))

    space_mask = ((np.abs(flon) >= space_low_cut) &
                  (np.abs(flon) <= space_high_cut) & (flon <= 0))  

    time_mask = time_mask[ :,None, None, None, None]
    space_mask = space_mask[None,None, None, None, :]

        
    mask = time_mask & space_mask
    fft_data_filtered = fft_data * mask
    filtered_data = np.real(ifft2(fft_data_filtered, axes=(0, 4)))

    return filtered_data

def detrending(data):
    mean_val = data.mean(dim="time")
    trend = data.polyfit(dim="time", deg=1)
    fitted = xr.polyval(data["time"], trend.polyfit_coefficients)
    detrended = data - fitted
    detrended += mean_val
    return detrended

In [ ]:
mods = ['aurora', 'pangu',]
for model in mods:
    for k in range(34):
    
        data = xr.open_dataset(f"/media/shrutee/Expansion/ai_models/new_lr/{model}_{k}.nc").sel(latitude= slice(21,-21))
        data = data.resample(time = '1D').mean('time')
        common_dims = list(data.dims.keys())
        common_coords = data.coords
        new_ds = []
        for i in list(data.data_vars):
            data_dtren = detrending(data[i])
            kv = space_time_filter(1/20,1/10,2/360,10/360,np.array(data_dtren.values),1,1)
            del data_dtren; gc.collect()
            kv_xr = xr.DataArray(data = kv,dims= common_dims,coords= common_coords, name = i)
            new_ds.append(kv_xr)
            del kv, kv_xr; gc.collect()
        ds = xr.merge(new_ds)
        # ds = ds.drop_vars('valid_time')
        ds.to_netcdf(f"/media/shrutee/Expansion/ai_models/new_lr/filtered/kw_{model}_{k}.nc")
        print(k)
        del ds; gc.collect()

# ROSSBY WAVE

In [ ]:
def space_time_filter(time_low_cut, time_high_cut,
                      space_low_cut, space_high_cut,
                      data, tim_int, space_int, ):

  
    nt, _, _, _, nl = data.shape
  
    ftime = fftfreq(nt, tim_int)            
    flon = fftfreq(nl, space_int)        
                     
    fft_data = fft2(data, axes=(0, 4))

    time_mask = ((np.abs(ftime) >= time_low_cut) &
                 (np.abs(ftime) <= time_high_cut) & (ftime > 0))

    space_mask = ((np.abs(flon) >= space_low_cut) &
                  (np.abs(flon) <= space_high_cut) & (flon >= 0))  

    time_mask = time_mask[ :,None, None, None, None]
    space_mask = space_mask[None,None, None, None, :]

          
    mask = time_mask & space_mask
    fft_data_filtered = fft_data * mask
    filtered_data = np.real(ifft2(fft_data_filtered, axes=(0, 4)))

    return filtered_data

def detrending(data):
    mean_val = data.mean(dim="time")
    trend = data.polyfit(dim="time", deg=1)
    fitted = xr.polyval(data["time"], trend.polyfit_coefficients)
    detrended = data - fitted
    detrended += mean_val
    return detrended

In [ ]:
mods = ['aurora', 'pangu', 'fc2']
for model in mods:
    for k in range( 34):
    
        data = xr.open_dataset(f"/media/shrutee/Expansion/ai_models/new_lr/{model}_{k}.nc")
        data = data.resample(time = '1D').mean('time')
        common_dims = list(data.dims.keys())
        common_coords = data.coords
        new_ds = []
        for i in list(data.data_vars):
            data_dtren = detrending(data[i])
            rb = space_time_filter(1/48,1/12,2/360,5/360, np.array(data_dtren.values),1,1)
            del data_dtren; gc.collect()
            rb_xr = xr.DataArray(data = rb, dims= common_dims,coords= common_coords, name = i)
            new_ds.append(rb_xr)
            del rb, rb_xr; gc.collect()
        ds = xr.merge(new_ds)
        # ds = ds.drop_vars('valid_time')
        ds.to_netcdf(f"/media/shrutee/Expansion/ai_models/new_lr/filtered/rb_{model}_{k}.nc")
        print(k)
        del ds; gc.collect()
        